# Week 16: Evaluate RAG as Separate Components

This notebook follows the reviewed Week 16 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Evaluate RAG as Separate Components
2. An Evaluation Dataset Represents Real Questions
3. Retrieval Precision and Recall Measure Different Goals
4. Top-k and Filters Change the Candidate Set
5. Query Rewriting Can Improve Retrieval
6. Hybrid Search Combines Exact and Semantic Retrieval
7. Reranking Applies a Stronger Model to Fewer Candidates
8. Answer Quality Has Several Dimensions
9. Model Judges Need Human Calibration
10. Failure Analysis Produces the Next Experiment
11. An Experiment Report Includes Quality, Latency, and Cost
12. Guided Lab: Improve RAG with Evidence

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Evaluate RAG as Separate Components

Evaluate at four boundaries:

1. **ingestion:** was source content recovered correctly?
2. **retrieval:** were relevant authorized chunks returned?
3. **context assembly:** was useful evidence preserved within budget?
4. **generation:** is the answer relevant, grounded, and correctly cited?

End-to-end correctness matters, but component metrics explain what to fix.

### Work it out first

The answer is wrong because the required chunk ranked `8` and top-k was `5`.

Generation cannot use evidence it never received. The retrieval stage owns this failure even if the final answer is the visible symptom.

### Notebook bridge

The advanced RAG and evaluation notebooks provide candidate retrieval and generation experiments.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
evaluation = {
    "retrieved_ids": retrieved_ids,
    "expected_ids": expected_ids,
    "answer": answer,
    "citation_ids": citation_ids,
}

Expected output:

```text
One evaluation record linking retrieval and answer evidence.
```


## 2. An Evaluation Dataset Represents Real Questions

A RAG evaluation example should record:

- question;
- user or tenant context;
- expected relevant chunk IDs;
- reference answer or required facts when available;
- unsupported or denied expectation;
- importance and category;
- notes about ambiguity.

Include normal, rare, multi-chunk, exact-term, paraphrase, unsupported, and access-control cases.

### Work it out first

Question: `How long do I have to request a refund?`

Expected chunk: `policy-v3-refunds-p7`  
Required fact: `14 days`  
Allowed outcome: supported answer with that citation  
Denied outcome: any answer using another tenant's policy

### Notebook bridge

Learners construct a small fixed dataset before tuning retrieval parameters.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
case = {
    "question": "How long do I have to request a refund?",
    "expected_chunk_ids": ["policy-v3-refunds-p7"],
    "required_facts": ["14 days"],
}

Expected output:

```text
A versioned test case with retrieval and answer expectations.
```


## 3. Retrieval Precision and Recall Measure Different Goals

For one query:

`retrieval precision = relevant retrieved / retrieved`

`retrieval recall = relevant retrieved / all relevant`

Precision asks how focused the results are. Recall asks how much required evidence was found.

The definition of relevant must come from labelled evidence, not from the retriever's own score.

### Work it out first

Expected relevant chunks `{A,B,C}`.  
Retrieved chunks `{A,B,X,Y}`.

Relevant retrieved: `{A,B}` count `2`

Precision `= 2/4 = 0.50`  
Recall `= 2/3 = 0.667`

### Notebook bridge

Learners calculate precision and recall at several k values.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
expected = {"A", "B", "C"}
retrieved = ["A", "B", "X", "Y"]
hits = expected.intersection(retrieved)
print(len(hits)/len(retrieved), len(hits)/len(expected))

Expected output:

```text
0.5 0.6666666666666666
```


## 4. Top-k and Filters Change the Candidate Set

`k` controls how many results enter the next stage.

Metadata filters can restrict:

- tenant or user access;
- document type;
- publication status;
- date;
- product or region.

Evaluate security filters as hard requirements. Tune relevance filters and k on the evaluation dataset, measuring quality, latency, token use, and duplicate rate.

### Work it out first

At `k=3`: precision `0.67`, recall `0.50`  
At `k=8`: precision `0.38`, recall `1.00`

Higher k recovers all evidence but adds more irrelevant chunks. Reranking or better queries may improve the tradeoff.

### Notebook bridge

Learners run a small top-k experiment before changing models.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
for k in [3, 5, 8]:
    results = retrieve(query, k=k, filters=access_filter)
    record_metrics(k, results, expected)

Expected output:

```text
A comparable result row for each k under the same access filter and dataset.
```


## 5. Query Rewriting Can Improve Retrieval

A **query rewrite** transforms a user's question into a search query.

Possible methods:

- resolve conversation references;
- expand abbreviations;
- generate alternate wording;
- split a multi-part question;
- add domain identifiers.

Rewriting can improve recall but may change intent. Preserve the original question and record every rewritten query.

### Work it out first

Conversation:

User: `What is the refund period?`  
Later: `Does it apply to annual plans?`

Standalone rewrite:

`Does the 14-day refund period apply to annual plans?`

The rewrite should be checked against conversation state, not invented from unrelated context.

### Notebook bridge

Advanced RAG experiments can compare original and rewritten query retrieval.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
rewrite = rewrite_model.invoke({
    "history": history,
    "question": question,
})
trace["rewritten_query"] = rewrite

Expected output:

```text
A standalone search query recorded beside the original user question.
```


## 6. Hybrid Search Combines Exact and Semantic Retrieval

Lexical search such as **BM25** is strong for exact terms, names, codes, and rare words.

Vector search is strong for semantic similarity and paraphrases.

**Hybrid search** combines candidate lists from both systems.

Raw lexical and vector scores have different meanings. **Reciprocal rank fusion (RRF)** combines ranks:

`RRF(d) = Σ 1 / (c + rank_i(d))`

### Work it out first

Document A ranks `1` lexical and `4` vector.  
With `c=60`:

`1/61 + 1/64 ≈ 0.01639 + 0.01563 = 0.03202`

Documents found by both methods gain combined evidence.

### Notebook bridge

Learners add a lexical candidate list before reranking.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
def rrf_score(ranks, c=60):
    return sum(1 / (c + rank) for rank in ranks)

Expected output:

```text
A fusion score based on rank positions rather than incompatible raw scores.
```


## 7. Reranking Applies a Stronger Model to Fewer Candidates

A **reranker** scores query-document pairs using a model more expensive than first-stage retrieval.

Pipeline:

1. retrieve a broader candidate set quickly;
2. rerank those candidates;
3. keep the best few for context.

Reranking can improve ordering but adds latency and cost. It cannot recover a relevant chunk missing from the candidate set.

### Work it out first

Vector search retrieves `20` candidates. Relevant chunk ranks `11`.  
Reranker moves it to rank `2`.  
Final context keeps top `5`, so the evidence is now included.

If the chunk was not in the first `20`, reranking could not help.

### Notebook bridge

The advanced RAG notebook includes a reranking stage before question answering.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
candidates = retriever.invoke(query)[:20]
reranked = reranker.rerank(query, candidates)
context_docs = reranked[:5]

Expected output:

```text
Five final chunks selected from twenty first-stage candidates.
```


## 8. Answer Quality Has Several Dimensions

- **Correctness:** agreement with a reference answer or verified facts
- **Relevance:** whether the response addresses the question
- **Groundedness:** whether claims are supported by supplied context
- **Citation correctness:** whether cited passages support nearby claims
- **Completeness:** whether required answer parts are present

An answer can be relevant but incorrect, correct by chance but ungrounded, or grounded but incomplete.

### Work it out first

Question asks refund period and exclusions.

Answer gives `14 days` with support but omits annual-plan exclusions.

It may be grounded and partly correct, but incomplete.

### Notebook bridge

The evaluation notebook builds metrics and a report across several answer dimensions.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
scores = {
    "correctness": 1.0,
    "relevance": 1.0,
    "groundedness": 1.0,
    "completeness": 0.5,
}

Expected output:

```text
Separate dimensions reveal the omitted requirement.
```


## 9. Model Judges Need Human Calibration

An **LLM judge** applies a rubric to an input, response, evidence, or reference answer.

Benefits:

- scalable evaluation of nuanced language;
- structured explanations;
- faster experiment comparison.

Risks:

- bias toward style or length;
- sensitivity to prompt and order;
- inconsistent scoring;
- shared errors with the evaluated model;
- cost and privacy concerns.

Calibrate judges against human-labelled examples and inspect disagreements.

### Work it out first

On `50` examples, judge and human agree on `42`.

Agreement rate:

`42 / 50 = 0.84`

The `8` disagreements should be grouped by cause before trusting the judge at scale.

### Notebook bridge

The evaluation notebook's critic model is compared with a manual rubric.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
agreement = sum(j == h for j, h in zip(judge_labels, human_labels))
print(agreement / len(human_labels))

Expected output:

```text
0.84
```


## 10. Failure Analysis Produces the Next Experiment

For each failed query, record:

- source and parsing status;
- expected and retrieved chunks;
- ranks and filters;
- rewrite;
- context after assembly;
- answer and citations;
- metric and rubric failures;
- owning stage;
- proposed change.

Group failures such as exact-term miss, multi-chunk miss, duplicate context, unauthorized retrieval, unsupported claim, or incorrect citation.

### Work it out first

Ten failed queries:

- 4 exact product codes missed by vector search;
- 3 relevant chunks ranked below k;
- 2 unsupported answers;
- 1 incorrect citation.

First experiment: add lexical retrieval for product-code cases, not rewrite every prompt.

### Notebook bridge

Learners build a failure report from evaluation traces.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
failures.groupby(["owning_stage", "category"]).size()

Expected output:

```text
Counts showing which stage and failure type dominate.
```


## 11. An Experiment Report Includes Quality, Latency, and Cost

Record for each RAG version:

- corpus and ingestion version;
- embedding model and index settings;
- retriever, filters, k, rewrite, and reranker;
- prompt and generation model;
- evaluation dataset version;
- component and end-to-end metrics;
- latency percentiles;
- input and output tokens;
- estimated cost;
- security failures;
- known limitations.

Do not release solely because one average improves.

### Work it out first

Version B improves retrieval recall from `0.72` to `0.84`, but p95 latency rises from `1.2s` to `4.8s`.

If acceptance requires p95 under `2s`, Version B does not pass despite quality improvement.

### Notebook bridge

Learners produce one baseline and two controlled RAG experiment reports.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
accepted = (
    recall >= 0.80
    and groundedness >= 0.95
    and p95_latency_ms <= 2000
    and security_failures == 0
)

Expected output:

```text
True only when every release threshold passes.
```


## 12. Guided Lab: Improve RAG with Evidence

Create at least `20` reviewed queries and:

1. label expected chunk IDs and required facts;
2. calculate retrieval precision and recall;
3. compare at least three k values;
4. test metadata filters;
5. compare original and rewritten queries;
6. add lexical retrieval and rank fusion;
7. rerank a fixed candidate set;
8. score correctness, relevance, groundedness, and citations;
9. calibrate one model judge with human labels;
10. classify every failure by owning stage;
11. compare latency, tokens, and cost;
12. choose a release candidate against explicit thresholds.

### Work it out first

Do not accept “Version B feels better.” Report:

`recall +0.12`, `groundedness unchanged`, `p95 +3.6s`, `cost +40%`, and the decision based on requirements.

### Notebook bridge

Complete `07.advanced-rag-with-llama-3-in-langchain.ipynb` and `12.llm-evaluation.ipynb`.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
results = evaluate_versions(
    versions=[baseline, hybrid, reranked],
    dataset=test_cases,
)
print(results.groupby("version").mean(numeric_only=True))

Expected output:

```text
Comparable component and end-to-end metrics for each version.
```


## Guided lab

Create at least `20` reviewed queries and:

1. label expected chunk IDs and required facts;
2. calculate retrieval precision and recall;
3. compare at least three k values;
4. test metadata filters;
5. compare original and rewritten queries;
6. add lexical retrieval and rank fusion;
7. rerank a fixed candidate set;
8. score correctness, relevance, groundedness, and citations;
9. calibrate one model judge with human labels;
10. classify every failure by owning stage;
11. compare latency, tokens, and cost;
12. choose a release candidate against explicit thresholds.

### Reference result

Do not accept “Version B feels better.” Report:

`recall +0.12`, `groundedness unchanged`, `p95 +3.6s`, `cost +40%`, and the decision based on requirements.


In [ ]:
# Guided lab workspace: Week 16
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://docs.langchain.com/langsmith/evaluate-rag-tutorial>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/12.llm-evaluation.ipynb>
- <https://docs.langchain.com/langsmith/evaluation-concepts>
- <https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-of-unranked-retrieval-sets-1.html>
- <https://python.langchain.com/docs/concepts/retrievers/>
- <https://python.langchain.com/docs/tutorials/rag/#query-analysis>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/07.advanced-rag-with-llama-3-in-langchain.ipynb>
- <https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion>
- <https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html>
- <https://www.sbert.net/examples/cross_encoder/applications/README.html>
- <https://www.nist.gov/itl/ai-risk-management-framework>
- <https://docs.langchain.com/langsmith/evaluation>
- <https://docs.langchain.com/langsmith/observability>